# A/B Test vs Bandit: See the Difference in 3 Minutes

**Goal:** Understand why multi-armed bandits beat traditional A/B testing.

**The Problem:** A/B tests waste traffic on inferior options while collecting data. Bandits adapt in real-time.

**Scenario:** You're testing two commodity trading signals. One has 4% win rate, the other 6%. Which approach wastes less opportunity?

---

In [ ]:
learning_objectives(["**Smaller difference:** Try `conversion_rate_A = 0.045` and `conversion_rate_B = 0.050` (very close!)", "**Larger difference:** Try `conversion_rate_A = 0.02` and `conversion_rate_B = 0.10` (obvious winner)", "**More traffic:** Change `n_visitors = 10000` - does the bandit's advantage grow?", "**Less traffic:** Change `n_visitors = 1000` - can the bandit still learn?"])

## Setup

In [ ]:
import sys; sys.path.insert(0, '../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
!pip install -q numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

In [ ]:
apply_course_theme()
apply_plot_theme()

## Scenario: Two Trading Signals

We have 5,000 trades to allocate between two signals:
- **Signal A:** 4% win rate (worse)
- **Signal B:** 6% win rate (better)

We don't know which is better at the start. Let's compare two approaches.

In [ ]:
# Ground truth (unknown to our algorithms)
conversion_rate_A = 0.04
conversion_rate_B = 0.06
n_visitors = 5000

print(f"Testing with {n_visitors} trades")
print(f"Signal A: {conversion_rate_A:.1%} win rate (worse)")
print(f"Signal B: {conversion_rate_B:.1%} win rate (better)")
print(f"\nDifference: {(conversion_rate_B - conversion_rate_A):.1%}")

## Approach 1: Traditional A/B Test

Split traffic 50/50 for the entire experiment. Count conversions at the end.

In [ ]:
# A/B Test: Fixed 50/50 split
n_A = n_visitors // 2
n_B = n_visitors // 2

# Simulate conversions
conversions_A = np.random.binomial(1, conversion_rate_A, n_A)
conversions_B = np.random.binomial(1, conversion_rate_B, n_B)

# Combine for plotting
ab_conversions = np.concatenate([conversions_A, conversions_B])
ab_cumulative = np.cumsum(ab_conversions)

total_conversions_ab = conversions_A.sum() + conversions_B.sum()

print("\n=== A/B Test Results ===")
print(f"Signal A: {n_A} trades → {conversions_A.sum()} wins ({100*conversions_A.sum()/n_A:.2f}%)")
print(f"Signal B: {n_B} trades → {conversions_B.sum()} wins ({100*conversions_B.sum()/n_B:.2f}%)")
print(f"Total conversions: {total_conversions_ab}")

## Approach 2: Thompson Sampling Bandit

Start with 50/50, but adapt as we learn which signal is better.

In [ ]:
# Thompson Sampling
np.random.seed(42)  # Same random seed for fair comparison

alpha = np.array([1.0, 1.0])  # Beta prior: successes + 1
beta = np.array([1.0, 1.0])   # Beta prior: failures + 1

conversions_bandit = []
arm_choices = []

for t in range(n_visitors):
    # Sample from posterior for each arm
    theta_A = np.random.beta(alpha[0], beta[0])
    theta_B = np.random.beta(alpha[1], beta[1])
    
    # Choose arm with higher sample
    chosen_arm = 0 if theta_A > theta_B else 1
    arm_choices.append(chosen_arm)
    
    # Get conversion (simulate trade outcome)
    true_rate = conversion_rate_A if chosen_arm == 0 else conversion_rate_B
    conversion = 1 if np.random.random() < true_rate else 0
    conversions_bandit.append(conversion)
    
    # Update beliefs
    if conversion == 1:
        alpha[chosen_arm] += 1
    else:
        beta[chosen_arm] += 1

bandit_cumulative = np.cumsum(conversions_bandit)
total_conversions_bandit = sum(conversions_bandit)

print("\n=== Thompson Sampling Results ===")
print(f"Signal A: {arm_choices.count(0)} trades → {sum([conversions_bandit[i] for i in range(len(arm_choices)) if arm_choices[i] == 0])} wins")
print(f"Signal B: {arm_choices.count(1)} trades → {sum([conversions_bandit[i] for i in range(len(arm_choices)) if arm_choices[i] == 1])} wins")
print(f"Total conversions: {total_conversions_bandit}")

## Side-by-Side Comparison

Cumulative conversions over time. The bandit adapts and pulls ahead!

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Cumulative conversions
ax1.plot(ab_cumulative, label='A/B Test (50/50 split)', linewidth=2, alpha=0.8)
ax1.plot(bandit_cumulative, label='Thompson Sampling', linewidth=2, alpha=0.8)
ax1.set_xlabel('Number of Trades', fontsize=12)
ax1.set_ylabel('Cumulative Wins', fontsize=12)
ax1.set_title('Cumulative Wins Over Time', fontsize=14)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Plot 2: Arm selection over time (bandit only)
window = 100
arm_B_pct = [100 * sum(arm_choices[max(0, i-window):i+1]) / min(window, i+1) 
             for i in range(len(arm_choices))]

ax2.plot(arm_B_pct, linewidth=2, color='green', alpha=0.7)
ax2.axhline(50, color='red', linestyle='--', label='A/B test (always 50%)', linewidth=2)
ax2.set_xlabel('Number of Trades', fontsize=12)
ax2.set_ylabel('% Traffic to Signal B (better option)', fontsize=12)
ax2.set_title('Thompson Sampling Learns to Prefer Signal B', fontsize=14)
ax2.set_ylim([0, 100])
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Calculate Wasted Traffic

How many trades did A/B testing "waste" on the inferior signal?

In [ ]:
# Wasted traffic = trades sent to worse option
ab_wasted = n_A  # A/B sent exactly half to worse option
bandit_wasted = arm_choices.count(0)  # Bandit sent this many to Signal A

# Extra conversions from bandit
extra_conversions = total_conversions_bandit - total_conversions_ab

print("\n" + "="*50)
print("WASTED TRAFFIC ANALYSIS")
print("="*50)
print(f"\nA/B Test:")
print(f"  Sent to worse signal: {ab_wasted:,} trades ({100*ab_wasted/n_visitors:.1f}%)")
print(f"  Total wins: {total_conversions_ab}")

print(f"\nThompson Sampling:")
print(f"  Sent to worse signal: {bandit_wasted:,} trades ({100*bandit_wasted/n_visitors:.1f}%)")
print(f"  Total wins: {total_conversions_bandit}")

print(f"\n" + "="*50)
print(f"BANDIT ADVANTAGE: {extra_conversions} more wins (+{100*extra_conversions/total_conversions_ab:.1f}%)")
print(f"Traffic saved: {ab_wasted - bandit_wasted:,} fewer trades to worse option")
print("="*50)

print(f"\n💡 Key insight: Bandit adapted in real-time, avoiding the worse option!")

## Modify This

Experiment with different scenarios:

1. **Smaller difference:** Try `conversion_rate_A = 0.045` and `conversion_rate_B = 0.050` (very close!)
2. **Larger difference:** Try `conversion_rate_A = 0.02` and `conversion_rate_B = 0.10` (obvious winner)
3. **More traffic:** Change `n_visitors = 10000` - does the bandit's advantage grow?
4. **Less traffic:** Change `n_visitors = 1000` - can the bandit still learn?

**Question to explore:** At what point is the difference too small for Thompson Sampling to detect?

In [ ]:
# YOUR EXPERIMENTS HERE
# Copy the code above and modify the parameters

## What's Next?

You just saw why bandits beat A/B tests: **they adapt while learning**.

**Continue learning:**
- `00_your_first_bandit.ipynb` - Learn Thompson Sampling basics
- `02_commodity_allocation_starter.ipynb` - Apply bandits to real commodity data
- `03_creator_bandit_playbook.ipynb` - Full playbook for content publishing
- `04_algorithm_comparison.ipynb` - Compare 4 different bandit algorithms

**Deep dive:**
- Module 1: Understanding regret and the explore-exploit tradeoff
- Module 3: When to use A/B tests vs bandits (bandits aren't always better!)
- Module 4: Contextual bandits for personalized decisions

**Key takeaway:** A/B tests answer "Which is better?" Bandits answer "Which should I use right now?" Different goals!

In [ ]:
key_takeaways(["- `00_your_first_bandit.ipynb` - Learn Thompson Sampling basics", "`02_commodity_allocation_starter.ipynb` - Apply bandits to real commodity data", "`03_creator_bandit_playbook.ipynb` - Full playbook for content publishing", "`04_algorithm_comparison.ipynb` - Compare 4 different bandit algorithms", "- Module 1: Understanding regret and the explore-exploit tradeoff", "Module 3: When to use A/B tests vs bandits (bandits aren't always better!)"])